<a href="https://colab.research.google.com/github/stephleao/wmc-desafio-streaming-churn/blob/main/%5BNina_da_Hora%5D_Probabilidade_e_Amostragem_%7C_Bootcamp_Data_Analytics_2026_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
import numpy as np

# 1. CARREGAMENTO DA BASE CORRIGIDO (Adicionado o sep=';')
url = 'https://raw.githubusercontent.com/stephleao/wmc-desafio-streaming-churn/refs/heads/main/clientes.csv'
df = pd.read_csv(url, sep=';')

print("=== 1. DIAGNÓSTICO INICIAL DOS DADOS ===")
print(f"Total de registros originais: {df.shape[0]} linhas e {df.shape[1]} colunas\n")

# 2. VERIFICAÇÃO DE SAÚDE DOS DADOS
print("--- Tipos de Dados ---")
print(df.dtypes)

print("\n--- Valores Ausentes (Nulos) por Coluna ---")
print(df.isna().sum())

print(f"\nLinhas duplicadas identificadas: {df.duplicated().sum()}")

# 3. LIMPEZA (Removendo duplicados, se existirem)
df = df.drop_duplicates()

# 4. CRIAÇÃO DOS GRUPOS COM BASE NO TEMPO DE ASSINATURA DOS CANCELADOS
# Consideramos quem cancelou (cancelou == 1 ou 'Sim') e mapeamos o tempo
# Adaptando a regra do negócio para as colunas reais da sua base:
condicoes = [
    (df['cancelou'].isin([1, 'Sim', 'sim', 'S']) & (df['tempo_assinatura_meses'] <= 6)),
    (df['cancelou'].isin([1, 'Sim', 'sim', 'S']) & (df['tempo_assinatura_meses'] > 24))
]
nomes_grupos = ['Ultimos 6 meses', 'Mais de 24 meses']

# Criando a nova coluna de segmentação
df['grupo_cancelamento'] = np.select(condicoes, nomes_grupos, default='Outros Periodos / Ativos')

print("\n=== 2. QUANTIDADE DE CLIENTES POR GRUPO ===")
print(df['grupo_cancelamento'].value_counts())

=== 1. DIAGNÓSTICO INICIAL DOS DADOS ===
Total de registros originais: 200 linhas e 7 colunas

--- Tipos de Dados ---
cliente_id                  int64
idade                       int64
tempo_assinatura_meses      int64
frequencia_uso_mensal       int64
regiao                        str
mensalidade               float64
cancelou                    int64
dtype: object

--- Valores Ausentes (Nulos) por Coluna ---
cliente_id                0
idade                     0
tempo_assinatura_meses    0
frequencia_uso_mensal     0
regiao                    0
mensalidade               0
cancelou                  0
dtype: int64

Linhas duplicadas identificadas: 0

=== 2. QUANTIDADE DE CLIENTES POR GRUPO ===
grupo_cancelamento
Outros Periodos / Ativos    159
Mais de 24 meses             32
Ultimos 6 meses               9
Name: count, dtype: int64


In [9]:
# Ver os nomes reais de todas as colunas da base
print(df.columns.tolist())

['cliente_id', 'idade', 'tempo_assinatura_meses', 'frequencia_uso_mensal', 'regiao', 'mensalidade', 'cancelou', 'grupo_cancelamento']


In [10]:
# ==============================
# IMPORTAÇÕES
# ==============================

import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

In [11]:
# ==========================================
# SEGMENTAÇÃO DA BASE POR STATUS DE CHURN
# ==========================================

# Objetivo:
# Filtra clientes que cancelaram o serviço (churn) e não churn
# Condição: cancelou == 1
# Condição: cancelou == 0

cancelaram = df[df['cancelou'] == 1]
ativos = df[df['cancelou'] == 0]

In [12]:
# ==========================================
# DISTRIBUIÇÃO DA VARIÁVEL ALVO (CHURN)
# ==========================================

# Objetivo:
# - entender o balanceamento da variável alvo (churn vs não churn)

display(Markdown("## Taxa de clientes por status de Churn"))
display(Markdown("---"))

percentual = (
    df['cancelou']
    .value_counts(normalize=True)
    .mul(100)
    .rename(index={0: 'Não Churn', 1: 'Churn'})
    .round(2)
    .reset_index()
)

# Ajuste de nomes das colunas para leitura mais clara
percentual.columns = ['Status', 'Percentual (%)']

# Exibição final da tabela formatada
display(percentual)

# Percentual de churn (linha onde Status == 'Churn')
churn_rate = percentual.loc[percentual['Status'] == 'Churn', 'Percentual (%)'].values[0]

display(Markdown("---"))
print(f'Percentual de Churn é de: {churn_rate:.2f}%')

## Taxa de clientes por status de Churn

---

,Status,Percentual (%)
0,Não Churn,72.0
1,Churn,28.0


---

Percentual de Churn é de: 28.00%


In [13]:
# ==========================================
# CRIAÇÃO DE VARIÁVEL CATEGÓRICA (GRUPO DE TEMPO)
# ==========================================

# Objetivo:
# - Facilitar análise de churn por estágio do cliente
# - Permitir comparação entre perfis de curto, médio e longo prazo

df['grupo_tempo'] = pd.cut(
    df['tempo_assinatura_meses'],  # variável contínua original
    bins=[0, 6, 24, df['tempo_assinatura_meses'].max()],  # limites das faixas
    labels=['Até 6 meses', '7 a 24 meses', 'Acima de 24 meses']  # categorias finais
)

In [14]:
# ==========================================
# CHURN POR GRUPO DE TEMPO (TABELA DE CONTINGÊNCIA)
# ==========================================

# Objetivo:
# - comparar risco de churn entre diferentes estágios de relacionamento do cliente

# Denominador:
# - total de clientes dentro de cada grupo de tempo

display(Markdown("## Tabela de contingência entre tempo de assinatura e status"))
display(Markdown("---"))

tabela = (
    pd.crosstab(
        df['grupo_tempo'],
        df['cancelou'],
        normalize='index'  
    )
    .mul(100) 
    .round(2)
    .rename(columns={
        0: 'Não churn (%)',
        1: 'Churn (%)'
    })
)

display(
    tabela.style
        .format("{:.2f}%")
        .background_gradient(cmap=sns.color_palette("flare", as_cmap=True))
)

## Tabela de contingência entre tempo de assinatura e status

---

cancelou,Não churn (%),Churn (%)
grupo_tempo,,
Até 6 meses,62.50%,37.50%
7 a 24 meses,71.15%,28.85%
Acima de 24 meses,74.19%,25.81%


In [15]:
# ==========================================
# ESTATÍSTICAS DESCRITIVAS DA BASE E SEGMENTAÇÃO POR CHURN
# ==========================================

display(Markdown("## Estatísticas descritivas da base"))
display(Markdown("---"))

# ------------------------------------------
# ESTATÍSTICAS GERAIS DA BASE
# ------------------------------------------

# Objetivo:
# - Entender o comportamento geral das variáveis numéricas
# - Identificar possíveis outliers e dispersões relevantes

display(df.describe().T.round(2))
display(Markdown("---"))

# ------------------------------------------
# PERFIL DOS CLIENTES EM CHURN
# ------------------------------------------
# - subconjunto da base onde cancelou == 1

display(Markdown("## Estatísticas descritivas dos clientes com churn"))
display(cancelaram.drop(columns=['cancelou']).describe().T.round(2))
display(Markdown("---"))

# ------------------------------------------
# PERFIL DOS CLIENTES ATIVOS
# ------------------------------------------
# - subconjunto da base onde cancelou == 0

display(Markdown("## Estatísticas descritivas dos clientes sem churn"))
display(ativos.drop(columns=['cancelou']).describe().T.round(2))

## Estatísticas descritivas da base

---

,count,mean,std,min,25%,50%,75%,max
cliente_id,200.0,100.50,57.88,1.00,50.75,100.50,150.25,200.00
idade,200.0,45.26,16.04,18.00,31.75,45.00,59.00,74.00
tempo_assinatura_meses,200.0,30.89,17.68,1.00,16.75,32.00,48.00,59.00
frequencia_uso_mensal,200.0,15.98,8.84,1.00,8.00,17.00,23.00,29.00
mensalidade,200.0,59.13,23.32,21.16,38.81,57.71,79.68,99.98
cancelou,200.0,0.28,0.45,0.00,0.00,0.00,1.00,1.00


---

## Estatísticas descritivas dos clientes com churn

,count,mean,std,min,25%,50%,75%,max
cliente_id,56.0,93.14,51.12,1.00,49.50,88.50,126.25,197.00
idade,56.0,43.29,16.21,18.00,28.75,42.50,56.00,74.00
tempo_assinatura_meses,56.0,29.66,18.79,1.00,14.25,29.50,49.00,59.00
frequencia_uso_mensal,56.0,15.71,9.08,1.00,6.75,15.00,25.00,29.00
mensalidade,56.0,57.33,23.41,21.16,37.84,53.64,79.83,99.98


---

## Estatísticas descritivas dos clientes sem churn

,count,mean,std,min,25%,50%,75%,max
cliente_id,144.0,103.36,60.23,2.00,51.75,104.50,155.50,200.00
idade,144.0,46.03,15.96,18.00,32.00,45.50,59.25,74.00
tempo_assinatura_meses,144.0,31.37,17.27,1.00,18.00,33.00,48.00,59.00
frequencia_uso_mensal,144.0,16.08,8.78,1.00,8.75,17.00,23.00,29.00
mensalidade,144.0,59.83,23.33,21.61,39.30,60.08,78.50,99.73


In [16]:
# ==========================================
# PERFIL MÉDIO: CHURN vs NÃO CHURN
# ==========================================

# Objetivo:
# - comparar o comportamento médio entre clientes que cancelam e os que permanecem
# - identificar variáveis com potencial poder explicativo para churn

# Denominador:
# - total de clientes dentro de cada classe de churn

display(Markdown("## Perfil médio: Churn vs Não churn"))
display(Markdown("---"))

perfil = df.groupby('cancelou')[[
    'idade',
    'tempo_assinatura_meses',
    'frequencia_uso_mensal',
    'mensalidade'
]].mean().T.round(2)

perfil.columns = ['Sem Churn', 'Churn']

display(
    perfil.style
        .format("{:.2f}")
        .background_gradient(
            cmap=sns.color_palette("flare", as_cmap=True),
            axis=None
        )
)

## Perfil médio: Churn vs Não churn

---

,Sem Churn,Churn
idade,46.03,43.29
tempo_assinatura_meses,31.37,29.66
frequencia_uso_mensal,16.08,15.71
mensalidade,59.83,57.33


In [17]:
# ==========================================
# TAXA DE CHURN POR REGIÃO E GRUPO DE TEMPO
# ==========================================

# Objetivo:
# - identificar se o churn varia conforme região e estágio do cliente
# - detectar interações entre fatores geográficos e comportamento do cliente

# Denominador:
# - total de clientes dentro de cada combinação (região × grupo_tempo)

display(Markdown("## Taxa de Churn por região vs Tempo de assinatura"))
display(Markdown("---"))

tabela = (
    df.groupby(['regiao', 'grupo_tempo'], observed=False)['cancelou']
    .mean()
    .mul(100)
    .round(2)
    .unstack()
)

tabela.columns.name = None
tabela.index.name = "Região"

display(
    tabela.style
        .format("{:.2f}%")
        .background_gradient(
            cmap=sns.color_palette("flare", as_cmap=True)
        )
)

## Taxa de Churn por região vs Tempo de assinatura

---

,Até 6 meses,7 a 24 meses,Acima de 24 meses
Região,,,
Centro-Oeste,57.14%,10.00%,23.33%
Nordeste,0.00%,55.56%,41.67%
Norte,20.00%,27.27%,20.00%
Sudeste,40.00%,30.77%,22.22%
Sul,nan%,22.22%,22.22%


In [18]:
# ==========================================
# CHURN POR REGIÃO (RISCO + IMPACTO)
# ==========================================

# Objetivo:
# - Comparar a taxa e participação de Churn de cada Região

# Participação no churn total:
# - mostra o peso de cada região dentro do total de churn da base

# Denominadores:
# - Clientes: total de registros na região
# - Churn: total de clientes da região (para a taxa via média)


display(Markdown("## Churn por Região"))
display(Markdown("---"))

regiao = df.groupby('regiao')['cancelou'].agg([
    ('Clientes', 'count'),
    ('Churn', 'sum'),
    ('Taxa churn', 'mean')
]).reset_index()

regiao['Taxa churn (%)'] = (regiao['Taxa churn'] * 100).round(2)

regiao['Participação churn (%)'] = (
    regiao['Churn'] / regiao['Churn'].sum() * 100
).round(2)

regiao = regiao.drop(columns=['Taxa churn'])

display(regiao.sort_values('Taxa churn (%)', ascending=False))

# Região com maior taxa de churn (risco)
maior_risco = regiao.loc[regiao['Taxa churn (%)'].idxmax(), ['regiao', 'Taxa churn (%)']]

# Região com maior participação no churn (impacto)
maior_impacto = regiao.loc[regiao['Participação churn (%)'].idxmax(), ['regiao', 'Participação churn (%)']]

display(Markdown("---"))
print(f"Maior taxa de churn: {maior_risco['regiao']} ({maior_risco['Taxa churn (%)']:.2f}%) - risco dentro da região")
print(f"Maior participação no churn: {maior_impacto['regiao']} ({maior_impacto['Participação churn (%)']:.2f}%)- impacto no total")

## Churn por Região

---

,regiao,Clientes,Churn,Taxa churn (%),Participação churn (%)
1,Nordeste,35,15,42.86,26.79
3,Sudeste,41,12,29.27,21.43
0,Centro-Oeste,47,12,25.53,21.43
4,Sul,36,8,22.22,14.29
2,Norte,41,9,21.95,16.07


---

Maior taxa de churn: Nordeste (42.86%) - risco dentro da região
Maior participação no churn: Nordeste (26.79%)- impacto no total


In [19]:
# Criação de variável categórica a partir da idade

df['faixa_etaria'] = pd.cut(
    df['idade'],
    bins=[0, 25, 35, 45, 60, 100],
    labels=['18-25', '26-35', '36-45', '46-60', '60+']
)

In [20]:
# ==========================================
# CHURN POR FAIXA ETÁRIA (INDEPENDENTE DO TEMPO)
# ==========================================

# Objetivo:
# - medir risco de churn isolando o efeito da idade
# - comparar comportamento entre diferentes faixas etárias sem interferência do tempo

# Denominador:
# - total de clientes dentro de cada faixa etária

display(Markdown("## Churn por faixa etária (independente do tempo)"))
display(Markdown("---"))

tabela = pd.crosstab(
    df['faixa_etaria'],
    df['cancelou'],
    normalize='index'
).mul(100).round(2)

tabela = tabela[[1]]
tabela.columns = ['Churn (%)']

tabela.index.name = "Faixa etária"

display(
    tabela.style
        .format("{:.2f}%")
        .background_gradient(cmap=sns.color_palette("flare", as_cmap=True))
)

display(Markdown("---"))
maior_churn = tabela['Churn (%)'].idxmax()
valor_maior_churn = tabela['Churn (%)'].max()

print(f"Maior taxa de churn por faixa etária: {maior_churn} ({valor_maior_churn:.2f}%)")

## Churn por faixa etária (independente do tempo)

---

,Churn (%)
Faixa etária,
18-25,35.48%
26-35,33.33%
36-45,25.00%
46-60,26.00%
60+,23.91%


---

Maior taxa de churn por faixa etária: 18-25 (35.48%)


In [21]:
# ==========================================
# CHURN POR FAIXA ETÁRIA E GRUPO DE TEMPO
# ==========================================

# Objetivo:
# - analisar a interação entre idade e tempo de relacionamento no risco de churn
# - identificar segmentos mais vulneráveis ao cancelamento

# Denominador:
# - total de clientes dentro de cada célula (faixa_etaria × grupo_tempo)

display(Markdown("## Churn por faixa etária e grupo de tempo"))
display(Markdown("---"))

tabela = pd.crosstab(
    df['faixa_etaria'],
    df['grupo_tempo'],
    values=df['cancelou'],
    aggfunc='mean',
    normalize=False
).mul(100).round(2)

tabela.columns.name = "Grupo de tempo"
tabela.index.name = "Faixa etária"

display(
    tabela.style
        .format("{:.2f}%")
        .background_gradient(cmap=sns.color_palette("flare", as_cmap=True))
)

maior_tempo = tabela.max().max()
grupo_maior_tempo = tabela.max().idxmax()
faixa_maior_tempo = tabela[grupo_maior_tempo].idxmax()

display(Markdown("---"))
print(f"Maior churn no grupo de tempo: {grupo_maior_tempo} + {faixa_maior_tempo} ({maior_tempo:.2f}%)")
display(Markdown("---"))

for col in tabela.columns:
    faixa = tabela[col].idxmax()
    valor = tabela[col].max()
    
    print(f"Maior churn em {col}: {faixa} ({valor:.2f}%)")

## Churn por faixa etária e grupo de tempo

---

Grupo de tempo,Até 6 meses,7 a 24 meses,Acima de 24 meses
Faixa etária,,,
18-25,66.67%,37.50%,30.00%
26-35,40.00%,57.14%,23.81%
36-45,25.00%,33.33%,23.33%
46-60,42.86%,17.65%,26.92%
60+,20.00%,21.43%,25.93%


---

Maior churn no grupo de tempo: Até 6 meses + 18-25 (66.67%)


---

Maior churn em Até 6 meses: 18-25 (66.67%)
Maior churn em 7 a 24 meses: 26-35 (57.14%)
Maior churn em Acima de 24 meses: 18-25 (30.00%)
